# 05 · Export for HLS (one checkpoint → every artifact)
Produces, from a single checkpoint and a **real** test window:
* `verification_data/*.txt` golden intermediates (`conv1d_out`, `l0_rope_out`, `l0_add_norm1_out`, `l0_add_norm2_out`, `pool_out`, `full_output`, …) and float weights `full_*.txt`
* `weights/*.h` quantised headers with the **same array names** the HLS already uses, plus `results.h`, `input_output.h`
* `rope_tables.h`, `config.h` (with new macros: `CONV_STRIDE`, `USE_ATTENTION`, `HAS_DELTA_HEAD`, `COND_PREV`, …), `types_suggested.h` (AP_RND, AP_SAT)
* `hil_verification_samples/` real windows + float outputs + manifest (replaces the inconsistent set in the repo)
* `normalization_constants.{json,h}`: per-channel EMG scale, glove min/max, mu, causal SOS filter, `DELTA_SCALE`
* `fixed_point_expectation.json`: the per-layer error a *correct* HLS build should reproduce

For the unchanged HLS (`variant=roformer`, `heads=absolute`, `cond_prev=False`) the headers are drop-in. Other variants need the HLS changes listed in `config.h` macros.

In [ ]:
# --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
import os, sys, glob, json
for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
    if os.path.exists(os.path.join(p, 'emg_common.py')):
        sys.path.insert(0, os.path.abspath(p)); break
else:
    raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
# processed data: output of 01_preprocess_split (same session) or attached as a dataset
candidates = [f'{WORK}/db2_processed'] + sorted(glob.glob('/kaggle/input/*/db2_processed')) + sorted(glob.glob('/kaggle/input/*'))
npz_files = []
for c in candidates:
    npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
    if npz_files: break
assert npz_files, 'no processed data found: run 01_preprocess_split first or attach its output'
print(len(npz_files), 'subjects:', [os.path.basename(os.path.dirname(f)) for f in npz_files])

In [ ]:
ckpts = sorted(glob.glob(f'{WORK}/outputs/*.pt') + glob.glob('/kaggle/input/*/outputs/*.pt') + glob.glob('/kaggle/input/*/*.pt'))
CKPT = os.environ.get('EMG_CKPT', ckpts[-1] if ckpts else None); assert CKPT, 'no checkpoint found'
model, cfg, extra = ec.load_checkpoint(CKPT); model.eval()
cfg.act_fmt, cfg.weight_fmt, cfg.rope_fmt = (10, 3), (10, 3), (8, 2)      # formats to quantise headers/tables with
cfg.export_weight_type = 'data_t'                                         # 'weight_t' once types.h has the typedef
EXPORT_DIR = f'{WORK}/hls_export/{os.path.splitext(os.path.basename(CKPT))[0]}'
print(CKPT, '| variant', cfg.variant, '| heads', cfg.heads, '| cond_prev', cfg.cond_prev, '| tokens', cfg.num_tokens)

In [ ]:
# ---- a real test window for the golden dumps + N real HIL windows spread over the test split ----
d = np.load(npz_files[0]); test_idx = np.where((d['split'] == 1) & (d['has_prev'] == 1))[0]
i = int(test_idx[len(test_idx) // 3])
sample_x = d['sEMG'][i].T.astype(np.float64)                       # [C, seq_len] channel-major = HLS input layout
sample_prev = d['angles_prev'][i].astype(np.float64) if cfg.cond_prev else None
N_HIL = int(os.environ.get('EMG_N_HIL', 40))
hil = [(d['sEMG'][j].T.astype(np.float64), d['angles_prev'][j].astype(np.float64) if cfg.cond_prev else None, int(d['restim'][j]), int(d['rep'][j]))
       for j in test_idx[np.linspace(0, len(test_idx) - 1, N_HIL).astype(int)]]
report = ec.export_hls(model, cfg, EXPORT_DIR, sample_x, sample_prev, hil, ec.load_norm_stats(npz_files[0]))
display(pd.DataFrame(report).T)
print(open(f'{EXPORT_DIR}/config.h').read()); print(open(f'{EXPORT_DIR}/types_suggested.h').read())

In [ ]:
# ---- package for download and print the copy instructions ----
import shutil; zip_path = shutil.make_archive(EXPORT_DIR, 'zip', EXPORT_DIR); print('archive:', zip_path)
print(f"""
Copy into the HLS project:
  {EXPORT_DIR}/weights/*.h            -> hls_code/weights/
  {EXPORT_DIR}/rope_tables.h          -> hls_code/rope_tables.h
  {EXPORT_DIR}/config.h               -> hls_code/config.h
  {EXPORT_DIR}/verification_data/     -> hls_code/verification_data/
  {EXPORT_DIR}/hil_verification_samples/ -> hil_verification_samples/
  {EXPORT_DIR}/normalization_constants.h -> vitis_workspace (PS firmware)
Then re-run C-simulation; per-layer cosine should match fixed_point_expectation.json (about 0.999).
HLS changes required by this checkpoint: """ + ', '.join(m for m, on in [('strided conv embedding', cfg.conv_stride > 1), ('no attention path', not cfg.use_attention),
      ('causal mask', cfg.causal), ('RMSNorm', cfg.norm == 'rmsnorm'), ('last-token pooling', cfg.pooling == 'last'),
      ('delta head output', hasattr(model, 'fc_delta')), ('prev-angle input + prev_w projection', cfg.cond_prev)] if on) or 'none (drop-in)')